# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulWasay65/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook audits the Week-5 Logistic Regression model using FlyRank's leakage and validation guidance. The goal is honest, directional decision-support — not causal proof or a guarantee of future ranking improvement.


## 1. Two paper findings + my methodology questions

### Finding 1 — The Content Results Curve

FlyRank reports that health score varies by content-age bucket, with the 61–90 day bucket near the peak and the 271–365 day bucket lower. The paper presents this as an observed pattern.

**My methodology question:** How exactly is the Health Score outcome constructed for each page and age bucket, and does the comparison separate content age from differences in site mix, content type, historical visibility, or other factors? I would want the outcome definition and validation design to be clear before interpreting the pattern beyond association.

I would treat this as a useful measured pattern, not evidence that content age itself causes ranking decline.

### Finding 2 — Click Capture by Position Tier

FlyRank reports weighted CTR of about 0.420% for the top-3 tier versus about 0.050% for the Deep tier, showing a large observed difference across position tiers.

**My methodology question:** Are position-tier assignments and the CTR measurement window defined independently enough to avoid circularity, and does the validation design support interpretation beyond the observed cross-sectional association? Could query mix, brand/non-brand traffic, device mix, or impression volume explain part of the gap?

I would use this as directional evidence for prioritizing already-visible pages, not as proof that moving a page to a higher tier will itself cause the reported CTR increase.

**Paper:** FlyRank, *The State of AI-Driven SEO — April 2026*. The paper explicitly frames its headline findings as patterns rather than proof of cause and effect.


In [1]:
# Section 1 — keep the two paper findings and methodology questions auditable.
paper_audit = {
    'finding_1': 'Content Results Curve: health score varies by content-age bucket.',
    'question_1': 'How is Health Score constructed, and does the comparison separate age from site/content-mix differences?',
    'finding_2': 'Click Capture by Position Tier: weighted CTR is much higher in top positions than deep positions.',
    'question_2': 'Are position tiers and CTR measured independently enough, and could query/device/brand mix explain part of the observed gap?'
}

for key, value in paper_audit.items():
    print(f'{key}: {value}')


finding_1: Content Results Curve: health score varies by content-age bucket.
question_1: How is Health Score constructed, and does the comparison separate age from site/content-mix differences?
finding_2: Click Capture by Position Tier: weighted CTR is much higher in top positions than deep positions.
question_2: Are position tiers and CTR measured independently enough, and could query/device/brand mix explain part of the observed gap?


## 2. My model under an honest split (before/after)

The Week-5 model predicts whether a page improves average position by at least 2 positions from March to April 2026. The decision-time features below come only from March.

**Before:** random stratified train/test split.

**After:** grouped split by `client_hash_id`, so clients in the test set are not represented in training. This tests generalization to unseen clients.

The same features, target, model family, and metrics are used for both splits. The base rate is reported beside the metrics.


In [2]:
# Section 2 — build the March-to-April frame and compare random vs grouped validation.

import duckdb
import pandas as pd
from google.colab import userdata
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

con = duckdb.connect()

HF_TOKEN = userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise ValueError('HF_TOKEN is missing. Add your Hugging Face token in Colab Secrets before running this cell.')

con.execute(
    'CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN ?)',
    [HF_TOKEN]
)

performance_path = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'

query = f'''
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        AVG(gsc_avg_position) AS march_avg_position
    FROM read_parquet('{performance_path}')
    WHERE CAST(month AS VARCHAR) LIKE '2026-03%'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS april_avg_position
    FROM read_parquet('{performance_path}')
    WHERE CAST(month AS VARCHAR) LIKE '2026-04%'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m.*,
    a.april_avg_position,
    CASE
        WHEN a.april_avg_position <= m.march_avg_position - 2 THEN 1
        ELSE 0
    END AS improved_2_positions
FROM march m
INNER JOIN april a USING (client_hash_id, content_hash_id)
WHERE m.march_avg_position IS NOT NULL
  AND a.april_avg_position IS NOT NULL
  AND m.march_impressions >= 1
'''

model_df = con.sql(query).df()

if model_df.empty:
    raise ValueError('The March-to-April modeling frame is empty. Check the warehouse path and month values.')

model_df['march_ctr'] = model_df['march_clicks'] / model_df['march_impressions'].clip(lower=1)

features = ['march_impressions', 'march_clicks', 'march_avg_position', 'march_ctr']
X = (
    model_df[features]
    .replace([float('inf'), -float('inf')], pd.NA)
    .dropna()
)
y = model_df.loc[X.index, 'improved_2_positions'].astype(int)
groups = model_df.loc[X.index, 'client_hash_id']

if y.nunique() < 2:
    raise ValueError('The target has only one class after cleaning; ROC-AUC and classification comparison cannot be computed.')

print('Model rows:', len(X))
print('Overall positive rate / base rate:', round(float(y.mean()), 4))
print('Features:', features)

def build_model():
    return Pipeline([
        ('scale', StandardScaler()),
        ('logreg', LogisticRegression(max_iter=1000, random_state=42))
    ])

# BEFORE: random stratified split.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

random_model = build_model()
random_model.fit(X_train, y_train)
pred_random = random_model.predict(X_test)
prob_random = random_model.predict_proba(X_test)[:, 1]

random_metrics = {
    'split': 'Random stratified',
    'accuracy': accuracy_score(y_test, pred_random),
    'roc_auc': roc_auc_score(y_test, prob_random),
    'test_positive_rate': y_test.mean(),
    'base_rate': y.mean(),
}

# AFTER: grouped split by client.
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

Xg_train, Xg_test = X.iloc[train_idx], X.iloc[test_idx]
yg_train, yg_test = y.iloc[train_idx], y.iloc[test_idx]

grouped_model = build_model()
grouped_model.fit(Xg_train, yg_train)
pred_grouped = grouped_model.predict(Xg_test)
prob_grouped = grouped_model.predict_proba(Xg_test)[:, 1]

grouped_metrics = {
    'split': 'Grouped by client',
    'accuracy': accuracy_score(yg_test, pred_grouped),
    'roc_auc': roc_auc_score(yg_test, prob_grouped),
    'test_positive_rate': yg_test.mean(),
    'base_rate': y.mean(),
}

comparison = pd.DataFrame([random_metrics, grouped_metrics])
display(comparison.round(4))
print('The grouped split is the more conservative estimate for generalization to unseen clients.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Model rows: 158549
Overall positive rate / base rate: 0.2283
Features: ['march_impressions', 'march_clicks', 'march_avg_position', 'march_ctr']


,split,accuracy,roc_auc,test_positive_rate,base_rate
0,Random stratified,0.7971,0.8146,0.2283,0.2283
1,Grouped by client,0.8714,0.7947,0.1170,0.2283


The grouped split is the more conservative estimate for generalization to unseen clients.


### Validation interpretation

The grouped result is the more honest deployment-style estimate for this task because the test clients are absent from training.

If grouped performance is lower than random-split performance, that gap is itself a useful finding: repeated client-specific patterns may have helped the random split.

Neither score proves that a content refresh causes ranking improvement.


## 3. Leakage audit

The final feature set is limited to March decision-time signals.

- March impressions, clicks, average position, and CTR are available before the April outcome window.
- April average position is used to construct the target, but is not used as a model feature.
- No product decision flag or existing-system score is included as an input.
- The timeline is **March features → April outcome**.

This checks the main leakage risks identified in the FlyRank validation guidance: label-derived features, future-window overlap, and decision-derived features.


In [3]:
# Section 3 — explicit leakage audit.

leakage_audit = pd.DataFrame([
    ['march_impressions', 'Low', 'March history; known before the April outcome window.', 'Keep'],
    ['march_clicks', 'Low', 'March history; known before the April outcome window.', 'Keep'],
    ['march_avg_position', 'Low', 'March average position; earlier than the April outcome.', 'Keep'],
    ['march_ctr', 'Low', 'Derived only from March clicks and impressions.', 'Keep'],
    ['april_avg_position', 'High if used as a feature', 'Future outcome used to create the label.', 'Target only; exclude from X'],
    ['product flags / existing scores', 'High if used', 'Could encode an existing decision rather than independent evidence.', 'Not used'],
], columns=['item', 'risk', 'reason', 'decision'])

display(leakage_audit)

future_columns_in_features = [c for c in features if c.startswith('april_')]
assert future_columns_in_features == [], (
    f'Future columns leaked into features: {future_columns_in_features}'
)

print('Leakage check passed: no April outcome column is present in the feature list.')
print('Timeline: March features -> April outcome.')


,item,risk,reason,decision
0,march_impressions,Low,March history; known before the April outcome ...,Keep
1,march_clicks,Low,March history; known before the April outcome ...,Keep
2,march_avg_position,Low,March average position; earlier than the April...,Keep
3,march_ctr,Low,Derived only from March clicks and impressions.,Keep
4,april_avg_position,High if used as a feature,Future outcome used to create the label.,Target only; exclude from X
5,product flags / existing scores,High if used,Could encode an existing decision rather than ...,Not used


Leakage check passed: no April outcome column is present in the feature list.
Timeline: March features -> April outcome.


## 3B. Real failure examples

These are real false-positive and false-negative examples from the grouped test set.

Only anonymous IDs and measured model fields are shown. The examples are not presented as causal explanations for why a page changed.


In [4]:
# Failure examples from the grouped test set.

error_df = model_df.loc[
    Xg_test.index,
    [
        'client_hash_id',
        'content_hash_id',
        'march_avg_position',
        'april_avg_position',
        'improved_2_positions',
    ],
].copy()

error_df['predicted_probability'] = prob_grouped
error_df['predicted_class'] = pred_grouped

false_positive = (
    error_df[
        (error_df['predicted_class'] == 1)
        & (error_df['improved_2_positions'] == 0)
    ]
    .sort_values('predicted_probability', ascending=False)
    .head(5)
)

false_negative = (
    error_df[
        (error_df['predicted_class'] == 0)
        & (error_df['improved_2_positions'] == 1)
    ]
    .sort_values('predicted_probability', ascending=True)
    .head(5)
)

print('False positives: predicted improvement, but the measured target was not achieved.')
display(false_positive[[
    'client_hash_id', 'content_hash_id', 'march_avg_position',
    'april_avg_position', 'predicted_probability'
]])

print('False negatives: predicted no improvement, but the measured target was achieved.')
display(false_negative[[
    'client_hash_id', 'content_hash_id', 'march_avg_position',
    'april_avg_position', 'predicted_probability'
]])


False positives: predicted improvement, but the measured target was not achieved.


,client_hash_id,content_hash_id,march_avg_position,april_avg_position,predicted_probability
71699,client_2094c6eb080311d5,content_9c8973511fcbe903,90.900000,90.406250,0.954849
141669,client_2094c6eb080311d5,content_882f5f906a960d69,88.525641,102.750000,0.948552
57306,client_2094c6eb080311d5,content_1487f15bc724fb6c,88.492381,87.148148,0.948410
76206,client_3f0ce4d44fe94f3d,content_b2645e4a1dfd57f8,86.248953,86.985563,0.941723
69078,client_2094c6eb080311d5,content_40109e1d46bdc3d5,85.878968,89.000000,0.940539


False negatives: predicted no improvement, but the measured target was achieved.


,client_hash_id,content_hash_id,march_avg_position,april_avg_position,predicted_probability
90387,client_e5c2aa26a8598242,content_c3f9ab6da78c2d75,7.359930,5.168948,0.088732
11429,client_e5c2aa26a8598242,content_d144d55fe6b6650b,11.346569,8.747914,0.093259
90308,client_e5c2aa26a8598242,content_46a0899f87260db2,10.641968,7.787903,0.106589
76336,client_2094c6eb080311d5,content_6e693f27f7503ccb,2.000000,0.000000,0.111019
63431,client_3f0ce4d44fe94f3d,content_0e3d5e0500d8c0b5,2.083333,0.000000,0.111488


## 4. Claim rewrite

### Earlier/bolder claim

> “The model can identify which pages will improve rankings after a content refresh.”

### Evidence-safe rewrite

> **Observed and measured:** the Week-5 Logistic Regression model provides **directional decision-support** for prioritizing pages for review using March search-performance signals against a measured March-to-April position-improvement target. The grouped-by-client validation is the more honest test of generalization to unseen clients. The model does **not** establish that a refresh causes improvement and should not be treated as a guarantee of future ranking movement.

The revised wording stays within the evidence and uses careful terms such as **observed, measured, directional, and decision-support**.


In [5]:
# Section 4 — compact claim record for the audit trail.

claim_record = {
    'old_claim': 'The model can identify which pages will improve rankings after a content refresh.',
    'safe_claim': (
        'The model provides directional decision-support for prioritizing pages '
        'for review against a measured March-to-April position-improvement target.'
    ),
    'causal_claim_supported': False,
    'guarantee_supported': False,
    'preferred_validation': 'Grouped by client',
}

for key, value in claim_record.items():
    print(f'{key}: {value}')


old_claim: The model can identify which pages will improve rankings after a content refresh.
safe_claim: The model provides directional decision-support for prioritizing pages for review against a measured March-to-April position-improvement target.
causal_claim_supported: False
guarantee_supported: False
preferred_validation: Grouped by client


## 5. Self-check

- [x] Two paper findings and constructive methodology questions are documented.
- [x] The model is compared under a random split and a grouped-by-client split.
- [x] Base rate is printed beside validation metrics.
- [x] Features are audited for label-derived and future-window leakage.
- [x] Real false-positive and false-negative examples are shown.
- [x] Claims are rewritten using careful language: observed, measured, directional, decision-support.
- [x] The notebook contains runtime checks for missing Hugging Face access, empty data, and one-class targets.

**Execution note:** Run the notebook top-to-bottom in Colab with the required `HF_TOKEN` secret, then save/commit the executed notebook.
